This Colab notebook includes all the code you need to experiment with page-level document classification using a RAG-based strategy. Upload your own multi-document pharmaceutical PDF, follow along step-by-step, and learn how to detect document boundaries, label each section by type, and generate clean metadata for smarter extraction and automation.


**Step 1: Extract Page-Level Content from Pharmaceutical PDF**





In [1]:
# ============================================
# STEP 1: Install Required Libraries
# ============================================
#
# WHAT WE'RE DOING:
# Installing PyPDF2, a Python library for reading and extracting
# text from PDF files.
#
# WHY THIS MATTERS:
# Pharmaceutical bundled files are multi-document PDFs. We need a
# reliable way to extract text from each page before we can
# classify or route documents.
#
# WHAT YOU'LL SEE:
# A brief installation log confirming PyPDF2 was installed.
# ============================================

!pip install -q PyPDF2

In [2]:
# ============================================
# STEP 2: Extract Page-Level Content from PDF
# ============================================
#
# WHAT WE'RE DOING:
# Loading the pharmaceutical blob PDF and extracting the raw
# text from every page into a list of dictionaries.
#
# WHY THIS MATTERS:
# Each page in a pharma blob file may belong to a different
# document (Certificate of Quality, BSE/TSE Declaration, etc.).
# Extracting per-page text is the foundation for classification.
#
# WHAT YOU'LL SEE:
# A list of dictionaries, each containing a page number and
# the extracted text content for that page.
# ============================================

from PyPDF2 import PdfReader

reader = PdfReader("pharma-blob-sample.pdf")
pages = [page.extract_text() for page in reader.pages]
doc_pages = [{"page_num": i, "text": p} for i, p in enumerate(pages)]
doc_pages

[{'page_num': 0,
  'text': 'Cytiva\n100 Results Way\nMarlborough, MA 01752\nUnited States\nPage 1 / 1\ncytiva.com\n3 June, 2022\nRe: AKTA ready Flow Kit Storage Conditions\nTo Whom It May Concern,\nThe recommended storage temperature for standard AKTA ready flow kits is provided in Section 8.3 of the Operating\nInstructions 28960345 and specified as > +5°C. This recommendation also applies to all modified AKTA ready flow kits\nas well, including the two listed in the below table. Extended storage below the recommended +5°C could lead to\nbrittleness or cracking of the plastic connectors. However, the operating temperature of AKTA ready flow kits is +2°C to\n+40°C. If the kits are allowed to acclimate to a warmer temperature before being used this would reduce the risk of\ndamage to the kit during setup and handling.\nDescription Part Number Operating Temperature\nHigh Flow Kit F, Modified, AKTA ready 29477427 +2°C to +40°C\nHigh Flow Gradient C, Modified, AKTA ready 29184612 +2°C to +4

**Step 2: Set Up Gemini LLM Connection**

In [8]:
# ============================================
# STEP 3: Set Up Gemini LLM Connection
# ============================================
#
# WHAT WE'RE DOING:
# Creating a helper function that sends a prompt to Google's
# Gemini 2.0 Flash model and returns the response text.
#
# WHY THIS MATTERS:
# We use an LLM to classify pharmaceutical document types and
# detect document boundaries -- tasks that require understanding
# natural language content on each page.
#
# WHAT YOU'LL SEE:
# No output -- this cell defines the function for later use.
# ============================================

def gemini_model(prompt):
    import google.generativeai as genai
    from google.colab import userdata

    genai.configure(api_key=userdata.get(''))

    model = genai.GenerativeModel("models/gemini-2.0-flash")
    response = model.generate_content(prompt)

    return response.text




from llama_cpp import Llama
import os

# Download Mistral model if not already present
model_path = "content/mistral.gguf"
if not os.path.exists(model_path):
    !wget https://huggingface.co/TheBloke/Mistral-7B-Instruct-v0.2-GGUF/resolve/main/mistral-7b-instruct-v0.2.Q4_K_M.gguf -O {model_path}
    print(f"Model downloaded to {model_path}")

# Verify file exists and check size
if os.path.exists(model_path):
    print(f"Model file exists. Size: {os.path.getsize(model_path) / (1024 * 1024):.2f} MB")
else:
    print("Model file not found!")

# Load the model with GPU acceleration
try:
    llm = Llama(
        model_path=model_path,
        n_gpu_layers=1,  # Start with 1 layer on GPU to be safe
        n_ctx=2048,      # Context window size
        verbose=True     # Show loading progress
    )

    print("Model loaded successfully!")



except Exception as e:
    print(f"Error loading model: {e}")


def gemini_model(prompt):
    #response = llm(prompt, max_tokens=256, temperature=0.1)
    response = llm.create_chat_completion(
    messages=[
        {"role": "user", "content": prompt}
    ]
)
    return response['choices'][0]['message']['content']

llama_model_loader: loaded meta data with 24 key-value pairs and 291 tensors from content/mistral.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = mistralai_mistral-7b-instruct-v0.2
llama_model_loader: - kv   2:                       llama.context_length u32              = 32768
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 4096
llama_model_loader: - kv   4:                          llama.block_count u32              = 32
llama_model_loader: - kv   5:                  llama.feed_forward_length u32              = 14336
llama_model_loader: - kv   6:                 llama.rope.dimension_count u32              = 128
llama_model_loader: - kv   7:                 llama.attention.h

Model file exists. Size: 4166.07 MB


create_tensor: loading tensor blk.2.ffn_down.scale
create_tensor: loading tensor blk.2.ffn_up.scale
create_tensor: loading tensor blk.2.attn_q.input_scale
create_tensor: loading tensor blk.2.attn_k.input_scale
create_tensor: loading tensor blk.2.attn_v.input_scale
create_tensor: loading tensor blk.2.attn_output.input_scale
create_tensor: loading tensor blk.2.ffn_gate.input_scale
create_tensor: loading tensor blk.2.ffn_down.input_scale
create_tensor: loading tensor blk.2.ffn_up.input_scale
create_tensor: loading tensor blk.3.attn_q.scale
create_tensor: loading tensor blk.3.attn_k.scale
create_tensor: loading tensor blk.3.attn_v.scale
create_tensor: loading tensor blk.3.attn_output.scale
create_tensor: loading tensor blk.3.ffn_gate.scale
create_tensor: loading tensor blk.3.ffn_down.scale
create_tensor: loading tensor blk.3.ffn_up.scale
create_tensor: loading tensor blk.3.attn_q.input_scale
create_tensor: loading tensor blk.3.attn_k.input_scale
create_tensor: loading tensor blk.3.attn_v.i

Model loaded successfully!


In [9]:
# ============================================
# STEP 4: Write Document Boundary Detection Function
# ============================================
#
# WHAT WE'RE DOING:
# Building a function that asks the LLM whether two consecutive
# pages belong to the same pharmaceutical document.
#
# WHY THIS MATTERS:
# Pharma blob files bundle multiple documents back-to-back.
# Detecting where one document ends and another begins is the
# critical first step before classification or extraction.
#
# WHAT YOU'LL SEE:
# True or False -- whether pages 1 and 2 (index 1 and 2) belong
# to the same document given the type "Certificate of Quality".
# ============================================

def is_same_document(prev_text, curr_text, doc_type=None):
    prompt = f"""
    You are checking whether two consecutive pages from a pharmaceutical
    document package belong to the SAME individual document.

    A NEW document starts when the page has:
    - A different document title or heading (e.g., "Certificate of Quality"
      vs "Packaging Specification" vs "Material Description Sheet")
    - A completely different topic or subject matter
    - Its own header with a new document number or reference

    Pages belong to the SAME document when:
    - The second page says "continued" or "page 2 of 2"
    - The content directly continues the previous page's discussion
    - They share the same document number or title

    Previous page type: {doc_type or 'unknown'}

    Previous Page:
    {prev_text}

    Current Page:
    {curr_text}

    Do these two pages belong to the SAME document? Answer ONLY 'Yes' or 'No'.
    """
    response = gemini_model(prompt)
    print(response)
    return response.strip().lower().startswith("yes")


prev_text = doc_pages[1]["text"]
curr_text = doc_pages[2]["text"]
doc_type = "Certificate of Quality"  # Optional, can be "unknown" or None

is_same_document(prev_text, curr_text, doc_type)

llama_perf_context_print:        load time =    7191.37 ms
llama_perf_context_print: prompt eval time =    7189.88 ms /   755 tokens (    9.52 ms per token,   105.01 tokens per second)
llama_perf_context_print:        eval time =    2448.62 ms /    55 runs   (   44.52 ms per token,    22.46 tokens per second)
llama_perf_context_print:       total time =    9645.54 ms /   810 tokens
llama_perf_context_print:    graphs reused =         53


 No.

The previous page is for the "AKTA ready Low Flow Kit" and the current page is for the "AKTA ready High Flow Kit". They have different product descriptions and different lot numbers and product article numbers. Therefore, they belong to different documents.


False

**Step 3: Write the Document Type Classifier**

In [10]:
# ============================================
# STEP 5: Write Document Type Classifier
# ============================================
#
# WHAT WE'RE DOING:
# Building a function that classifies the first page of each
# new document into a pharmaceutical document category.
#
# WHY THIS MATTERS:
# Once we detect a boundary, we need to label the new document.
# Accurate classification enables downstream routing -- e.g.,
# sending a Certificate of Quality to QA and a BSE/TSE
# Declaration to regulatory review.
#
# WHAT YOU'LL SEE:
# The predicted document type for a sample page.
# ============================================

def classify_document_type(text):
    prompt = f"""
    You are a pharmaceutical document classifier. Based on the page
    content below, classify it into ONE of these document types:

    - Cover Letter: A formal letter (often starting with "To Whom It
      May Concern") discussing product information or storage conditions.
    - Certificate of Quality: Contains lot numbers, manufacture dates,
      expiration dates, and test results (autoclave, gamma irradiation).
    - Packaging Specification: Describes packaging components, materials,
      part numbers, and configuration change history.
    - BSE/TSE Declaration: A declaration about animal-origin materials
      and transmissible spongiform encephalopathy compliance.
    - Material Description: Lists materials of construction, sterilization
      compatibility, and physical properties of a product.
    - Supplier Qualification: Contains supplier audit history,
      certifications (ISO 9001, ISO 13485), and approved product lists.
    - Chain of Custody: Lists manufactured assemblies, traceability
      information, and the manufacturing-to-shipment flow.
    - Other: Use ONLY if the content does not match any of the above.

    Page Content:
    {text}

    Respond with ONLY the document type name. No explanation.
    """
    response = gemini_model(prompt).strip().lower().replace(".", "")
    result = response.title()
    return result

classify_document_type(doc_pages[3]["text"])

Llama.generate: 9 prefix-match hit, remaining 579 prompt tokens to eval
llama_perf_context_print:        load time =    7191.37 ms
llama_perf_context_print: prompt eval time =    5366.06 ms /   579 tokens (    9.27 ms per token,   107.90 tokens per second)
llama_perf_context_print:        eval time =     156.76 ms /     4 runs   (   39.19 ms per token,    25.52 tokens per second)
llama_perf_context_print:       total time =    5524.79 ms /   583 tokens
llama_perf_context_print:    graphs reused =          3


'Packaging Specification'

**Step 4: Loop Through Pages and Generate Page-Level Metadata**

In [11]:
# ============================================
# STEP 6: Loop Through Pages and Generate Metadata
# ============================================
#
# WHAT WE'RE DOING:
# Iterating through every page in the PDF. For each page we
# check if it continues the previous document or starts a new
# one, then record its document ID and type.
#
# WHY THIS MATTERS:
# This is the core pipeline -- it produces the page-level
# metadata table that tells us exactly which pages belong to
# which pharmaceutical document and what type each document is.
#
# WHAT YOU'LL SEE:
# A printed list of dictionaries showing page number, document
# ID, and document type for every page in the blob file.
# ============================================

results = []
current_doc_type = None
doc_counter = 0

for i, page in enumerate(doc_pages):
    if i == 0:
        current_doc_type = classify_document_type(page["text"])
    else:
        prev_text = doc_pages[i - 1]["text"]
        same = is_same_document(prev_text, page["text"], current_doc_type)
        if not same:
            doc_counter += 1
            current_doc_type = classify_document_type(page["text"])

    results.append({
        "page": i,
        "doc_id": doc_counter,
        "doc_type": current_doc_type
    })


for r in results:
    print(r)

Llama.generate: 293 prefix-match hit, remaining 408 prompt tokens to eval
llama_perf_context_print:        load time =    7191.37 ms
llama_perf_context_print: prompt eval time =    3419.39 ms /   408 tokens (    8.38 ms per token,   119.32 tokens per second)
llama_perf_context_print:        eval time =     114.58 ms /     3 runs   (   38.19 ms per token,    26.18 tokens per second)
llama_perf_context_print:       total time =    3535.34 ms /   411 tokens
llama_perf_context_print:    graphs reused =          2
Llama.generate: 9 prefix-match hit, remaining 861 prompt tokens to eval
llama_perf_context_print:        load time =    7191.37 ms
llama_perf_context_print: prompt eval time =    7544.35 ms /   861 tokens (    8.76 ms per token,   114.13 tokens per second)
llama_perf_context_print:        eval time =    2607.56 ms /    66 runs   (   39.51 ms per token,    25.31 tokens per second)
llama_perf_context_print:       total time =   10158.28 ms /   927 tokens
llama_perf_context_print:   

 Based on the information provided, these two pages belong to DIFFERENT documents. The previous page is a cover letter regarding the storage conditions for AKTA ready flow kits, while the current page is a certificate of quality for a specific lot of AKTA ready low flow kit. Therefore, the answer is 'No'.


llama_perf_context_print:        load time =    7191.37 ms
llama_perf_context_print: prompt eval time =    4993.41 ms /   575 tokens (    8.68 ms per token,   115.15 tokens per second)
llama_perf_context_print:        eval time =     192.57 ms /     5 runs   (   38.51 ms per token,    25.96 tokens per second)
llama_perf_context_print:       total time =    5186.86 ms /   580 tokens
llama_perf_context_print:    graphs reused =          4
Llama.generate: 9 prefix-match hit, remaining 746 prompt tokens to eval
llama_perf_context_print:        load time =    7191.37 ms
llama_perf_context_print: prompt eval time =    6548.45 ms /   746 tokens (    8.78 ms per token,   113.92 tokens per second)
llama_perf_context_print:        eval time =    2143.67 ms /    55 runs   (   38.98 ms per token,    25.66 tokens per second)
llama_perf_context_print:       total time =    8697.18 ms /   801 tokens
llama_perf_context_print:    graphs reused =         53
Llama.generate: 9 prefix-match hit, remaining 

 No.

The previous page is for the "AKTA ready Low Flow Kit" and the current page is for the "AKTA ready High Flow Kit". They have different product descriptions and different lot numbers and product article numbers. Therefore, they belong to different documents.


llama_perf_context_print:        load time =    7191.37 ms
llama_perf_context_print: prompt eval time =    5078.51 ms /   575 tokens (    8.83 ms per token,   113.22 tokens per second)
llama_perf_context_print:        eval time =     196.68 ms /     5 runs   (   39.34 ms per token,    25.42 tokens per second)
llama_perf_context_print:       total time =    5276.05 ms /   580 tokens
llama_perf_context_print:    graphs reused =          4
Llama.generate: 9 prefix-match hit, remaining 750 prompt tokens to eval
llama_perf_context_print:        load time =    7191.37 ms
llama_perf_context_print: prompt eval time =    6693.46 ms /   750 tokens (    8.92 ms per token,   112.05 tokens per second)
llama_perf_context_print:        eval time =    1401.58 ms /    36 runs   (   38.93 ms per token,    25.69 tokens per second)
llama_perf_context_print:       total time =    8098.40 ms /   786 tokens
llama_perf_context_print:    graphs reused =         34
Llama.generate: 9 prefix-match hit, remaining 

...

 Based on the information provided, these two pages do not belong to the same document. The previous page is a Material Description Sheet for a specific product, while the current page is a Supplier Qualification Record for a supplier. They have different document titles and topics, and they do not share the same document number.


llama_perf_context_print:        load time =    7191.37 ms
llama_perf_context_print: prompt eval time =    6081.76 ms /   687 tokens (    8.85 ms per token,   112.96 tokens per second)
llama_perf_context_print:        eval time =     196.87 ms /     5 runs   (   39.37 ms per token,    25.40 tokens per second)
llama_perf_context_print:       total time =    6279.44 ms /   692 tokens
llama_perf_context_print:    graphs reused =          4
Llama.generate: 9 prefix-match hit, remaining 937 prompt tokens to eval
llama_perf_context_print:        load time =    7191.37 ms
llama_perf_context_print: prompt eval time =    8781.26 ms /   937 tokens (    9.37 ms per token,   106.70 tokens per second)
llama_perf_context_print:        eval time =    1028.52 ms /    25 runs   (   41.14 ms per token,    24.31 tokens per second)
llama_perf_context_print:       total time =    9812.35 ms /   962 tokens
llama_perf_context_print:    graphs reused =         24
Llama.generate: 184 prefix-match hit, remainin

 Yes. The current page indicates that it is a continuation of the previous page and shares the same document number and title.


llama_perf_context_print:        load time =    7191.37 ms
llama_perf_context_print: prompt eval time =    7107.46 ms /   696 tokens (   10.21 ms per token,    97.93 tokens per second)
llama_perf_context_print:        eval time =    1771.33 ms /    44 runs   (   40.26 ms per token,    24.84 tokens per second)
llama_perf_context_print:       total time =    8882.91 ms /   740 tokens
llama_perf_context_print:    graphs reused =         43


 Yes, these two pages belong to the same document based on the shared document number (CYTIVA-0847) and the continuation of the discussion from the previous page about the supplier qualification record.
{'page': 0, 'doc_id': 0, 'doc_type': 'Cover Letter'}
{'page': 1, 'doc_id': 1, 'doc_type': 'Certificate Of Quality'}
{'page': 2, 'doc_id': 2, 'doc_type': 'Certificate Of Quality'}
{'page': 3, 'doc_id': 3, 'doc_type': 'Packaging Specification'}
{'page': 4, 'doc_id': 3, 'doc_type': 'Packaging Specification'}
{'page': 5, 'doc_id': 4, 'doc_type': 'Bse/Tse Declaration'}
{'page': 6, 'doc_id': 5, 'doc_type': 'Material Description'}
{'page': 7, 'doc_id': 6, 'doc_type': 'Supplier Qualification'}
{'page': 8, 'doc_id': 6, 'doc_type': 'Supplier Qualification'}
{'page': 9, 'doc_id': 6, 'doc_type': 'Supplier Qualification'}


**Step 5: Visualize Results**

In [12]:
# ============================================
# STEP 7: Visualize Results
# ============================================
#
# WHAT WE'RE DOING:
# Loading the metadata into a pandas DataFrame for a clean,
# tabular view of the classification results.
#
# WHY THIS MATTERS:
# A DataFrame makes it easy to filter, sort, and export the
# results -- for example, pulling all pages that belong to
# "Certificate of Quality" documents for QA review.
#
# WHAT YOU'LL SEE:
# A table with columns: page, doc_id, doc_type showing the
# classification for every page in the pharmaceutical blob file.
# ============================================

import pandas as pd

df = pd.DataFrame(results)
df

,page,doc_id,doc_type
0,0,0,Cover Letter
1,1,1,Certificate Of Quality
2,2,2,Certificate Of Quality
3,3,3,Packaging Specification
4,4,3,Packaging Specification
5,5,4,Bse/Tse Declaration
6,6,5,Material Description
7,7,6,Supplier Qualification
8,8,6,Supplier Qualification
9,9,6,Supplier Qualification
